In [ ]:
import tkinter as tk
from tkinter import ttk, messagebox, filedialog
import hashlib
import json
import xml.etree.ElementTree as ET
import uuid


class DBManager:
    def __init__(self):
        self.authors = []
        self.books = []
        self.users = []
        h = hashlib.md5("admin".encode()).hexdigest()
        self.users.append({"username": "admin", "password_hash": h})

    def authenticate(self, user, pwd):
        h = hashlib.md5(pwd.encode()).hexdigest()
        for u in self.users:
            if u["username"] == user and u["password_hash"] == h:
                return True
        return False

    def get_authors(self):
        return [(a["_id"], a["name"], a["country"], a["years"]) for a in self.authors]

    def get_books(self):
        return [(b["_id"], b["author_id"], b["title"], b["pages"], b["publisher"], b["year"]) for b in self.books]

    def add_author(self, name, country, years):
        by = int(years.split('-')[0]) if '-' in years else (int(years) if years.isdigit() else 0)
        new_id = str(uuid.uuid4())[:8]
        self.authors.append({
            "_id": new_id,
            "name": name,
            "country": country,
            "years": years,
            "birth_year": by
        })

    def add_book(self, aid, title, pages, pub, year):
        new_id = str(uuid.uuid4())[:8]
        self.books.append({
            "_id": new_id,
            "author_id": aid,
            "title": title,
            "pages": int(pages),
            "publisher": pub,
            "year": int(year)
        })

    def run_queries(self, x, y, n):
        report = "--- NoSQL РЕЗУЛЬТАТЫ (Emulator) ---\n\n"

        a1 = [a["name"] for a in self.authors if x <= a["birth_year"] <= y]
        report += f"1. Авторы ({x}-{y}):\n" + ", ".join(a1) + "\n\n"

        ru_ids = [a["_id"] for a in self.authors if a["country"] == "Russia"]
        b1 = [b["title"] for b in self.books if b["author_id"] in ru_ids]
        report += "2. Книги авторов из Russia:\n" + ", ".join(b1) + "\n\n"

        b2 = [b["title"] for b in self.books if b["pages"] > n]
        report += f"3. Книги > {n} страниц:\n" + ", ".join(b2) + "\n\n"

        counts = {}
        for b in self.books:
            counts[b["author_id"]] = counts.get(b["author_id"], 0) + 1

        winners = []
        for aid, count in counts.items():
            if count > 1:
                for a in self.authors:
                    if a["_id"] == aid: winners.append(a["name"])

        report += "4. Авторы с более чем 1 книгой:\n" + ", ".join(winners)
        return report


class LoginWindow(tk.Toplevel):
    def __init__(self, parent, db, on_success):
        super().__init__(parent)
        self.db, self.on_success = db, on_success
        self.title("Вход (NoSQL)")
        self.geometry("300x200")
        tk.Label(self, text="Логин:").pack(pady=5)
        self.u = tk.Entry(self);
        self.u.pack()
        tk.Label(self, text="Пароль:").pack(pady=5)
        self.p = tk.Entry(self, show="*");
        self.p.pack()
        tk.Button(self, text="Войти", command=self.login, width=15).pack(pady=20)

    def login(self):
        if self.db.authenticate(self.u.get(), self.p.get()):
            self.on_success();
            self.destroy()
        else:
            messagebox.showerror("Ошибка", "Неверный вход")


class LibraryApp:
    def __init__(self, root, db):
        self.root, self.db = root, db
        self.root.title("Библиотека (NoSQL Emulator)")
        self.root.geometry("800x600")
        tabs = ttk.Notebook(root)
        self.t1, self.t2, self.t3 = ttk.Frame(tabs), ttk.Frame(tabs), ttk.Frame(tabs)
        tabs.add(self.t1, text="Авторы");
        tabs.add(self.t2, text="Книги");
        tabs.add(self.t3, text="Запросы")
        tabs.pack(expand=1, fill="both")
        self.setup_ui();
        self.refresh()

    def setup_ui(self):
        cols_a = ("ID", "Имя", "Страна", "Годы")
        self.tree_a = ttk.Treeview(self.t1, columns=cols_a, show='headings')
        for c in cols_a: self.tree_a.heading(c, text=c)
        self.tree_a.pack(expand=1, fill="both", padx=5, pady=5)
        btns_a = ttk.Frame(self.t1);
        btns_a.pack(fill="x", padx=5)
        ttk.Button(btns_a, text="Добавить", command=self.add_a).pack(side="left")
        ttk.Button(btns_a, text="Импорт", command=self.import_f).pack(side="left")
        ttk.Button(btns_a, text="Экспорт JSON", command=self.export_f).pack(side="left")

        cols_b = ("ID", "AID", "Название", "Стр", "Изд", "Год")
        self.tree_b = ttk.Treeview(self.t2, columns=cols_b, show='headings')
        for c in cols_b: self.tree_b.heading(c, text=c)
        self.tree_b.pack(expand=1, fill="both", padx=5, pady=5)
        ttk.Button(self.t2, text="Добавить книгу", command=self.add_b).pack()

        self.q_text = tk.Text(self.t3, height=20)
        self.q_text.pack(expand=1, fill="both", padx=10, pady=10)
        ttk.Button(self.t3, text="ВЫПОЛНИТЬ ЗАПРОСЫ", command=self.run_special).pack(pady=10)

    def refresh(self):
        for i in self.tree_a.get_children(): self.tree_a.delete(i)
        for r in self.db.get_authors(): self.tree_a.insert("", "end", values=r)
        for i in self.tree_b.get_children(): self.tree_b.delete(i)
        for r in self.db.get_books(): self.tree_b.insert("", "end", values=r)

    def run_special(self):
        res = self.db.run_queries(1800, 1900, 300)
        self.q_text.delete(1.0, tk.END)
        self.q_text.insert(tk.END, res)

    def add_a(self):
        w = tk.Toplevel(self.root)
        tk.Label(w, text="Имя").pack();
        e1 = tk.Entry(w);
        e1.pack()
        tk.Label(w, text="Страна").pack();
        e2 = tk.Entry(w);
        e2.pack()
        tk.Label(w, text="Годы").pack();
        e3 = tk.Entry(w);
        e3.pack()

        def save(): self.db.add_author(e1.get(), e2.get(), e3.get()); self.refresh(); w.destroy()

        tk.Button(w, text="ОК", command=save).pack(pady=10)

    def add_b(self):
        w = tk.Toplevel(self.root)
        tk.Label(w, text="ID автора (из табл)").pack();
        e1 = tk.Entry(w);
        e1.pack()
        tk.Label(w, text="Название").pack();
        e2 = tk.Entry(w);
        e2.pack()
        tk.Label(w, text="Стр").pack();
        e3 = tk.Entry(w);
        e3.pack()
        tk.Label(w, text="Изд").pack();
        e4 = tk.Entry(w);
        e4.pack()
        tk.Label(w, text="Год").pack();
        e5 = tk.Entry(w);
        e5.pack()

        def save(): self.db.add_book(e1.get(), e2.get(), e3.get(), e4.get(), e5.get()); self.refresh(); w.destroy()

        tk.Button(w, text="ОК", command=save).pack(pady=10)

    def export_f(self):
        s = self.tree_a.selection()
        if not s: return
        v = self.tree_a.item(s[0])['values']
        d = {"name": v[1], "country": v[2], "years": v[3]}
        p = filedialog.asksaveasfilename(defaultextension=".json")
        if p:
            with open(p, 'w', encoding='utf-8') as f: json.dump(d, f, ensure_ascii=False, indent=4)

    def import_f(self):
        p = filedialog.askopenfilename()
        if not p: return
        try:
            if p.endswith('.json'):
                with open(p, 'r', encoding='utf-8') as f:
                    d = json.load(f);
                    n, c, y = d['name'], d['country'], str(d['years'])
            else:
                r = ET.parse(p).getroot();
                n, c = r.find('name').text, r.find('country').text
                yn = r.find('years');
                y = f"{yn.get('born')}-{yn.get('died')}"
            self.db.add_author(n, c, y);
            self.refresh()
        except:
            messagebox.showerror("Ошибка", "Файл не подходит")


if __name__ == "__main__":
    db_manager = DBManager()
    main_root = tk.Tk();
    main_root.withdraw()


    def start(): main_root.deiconify(); LibraryApp(main_root, db_manager)


    LoginWindow(main_root, db_manager, start)
    main_root.mainloop()